In [1]:
import pandas as pd
import sqlite3

print("Initializing SQL Engine & Loading Cleaned Datasets...")

# 1. Connect to an in-memory SQL database
conn = sqlite3.connect(':memory:')

# 2. Load all clean CSV files from data/processed/
orders = pd.read_csv('../data/processed/clean_orders.csv')
items = pd.read_csv('../data/processed/clean_order_items.csv')
products = pd.read_csv('../data/processed/clean_products.csv')
reviews = pd.read_csv('../data/processed/clean_reviews.csv')
customers = pd.read_csv('../data/processed/clean_customers.csv')
sellers = pd.read_csv('../data/processed/clean_sellers.csv')
payments = pd.read_csv('../data/processed/clean_payments.csv')

# 3. Write DataFrames directly to SQL tables
orders.to_sql('clean_orders', conn, index=False, if_exists='replace')
items.to_sql('clean_order_items', conn, index=False, if_exists='replace')
products.to_sql('clean_products', conn, index=False, if_exists='replace')
reviews.to_sql('clean_reviews', conn, index=False, if_exists='replace')
customers.to_sql('clean_customers', conn, index=False, if_exists='replace')
sellers.to_sql('clean_sellers', conn, index=False, if_exists='replace')
payments.to_sql('clean_payments', conn, index=False, if_exists='replace')

# Load marketing funnel datasets
mql = pd.read_csv('../data/olist_marketing_qualified_leads_dataset.csv')
closed = pd.read_csv('../data/olist_closed_deals_dataset.csv')
mql.to_sql('olist_marketing_qualified_leads_dataset', conn, index=False, if_exists='replace')
closed.to_sql('olist_closed_deals_dataset', conn, index=False, if_exists='replace')

print("SUCCESS: All tables loaded into SQL memory! Ready to execute queries.")

Initializing SQL Engine & Loading Cleaned Datasets...
SUCCESS: All tables loaded into SQL memory! Ready to execute queries.


In [3]:
# Query 1: Extract Delivered Orders
q1 = """
SELECT 
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_estimated_delivery_date
FROM clean_orders
WHERE order_status = 'delivered'
ORDER BY order_purchase_timestamp DESC
LIMIT 5;
"""

pd.read_sql_query(q1, conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_estimated_delivery_date
0,35a972d7f8436f405b56e36add1a7140,898b7fee99c4e42170ab69ba59be0a8b,delivered,2018-08-29 15:00:37,2018-09-05
1,03ef5dedbe7492bdae72eec50764c43f,496630b6740bcca28fce9ba50d8a26ef,delivered,2018-08-29 14:52:00,2018-09-03
2,168626408cb32af0ffaf76711caae1dc,6e353700bc7bcdf6ebc15d6de16d7002,delivered,2018-08-29 14:18:28,2018-09-11
3,0b223d92c27432930dfe407c6aea3041,e60df9449653a95af4549bbfcb18a6eb,delivered,2018-08-29 14:18:23,2018-09-04
4,52018484704db3661b98ce838612b507,e450a297a7bc6839ceb0cf1a2377fa02,delivered,2018-08-29 12:25:59,2018-09-03


In [4]:
# Query 2: High-Value Product Transactions
q2 = """
SELECT 
    order_id,
    order_item_id,
    product_id,
    seller_id,
    price,
    freight_value
FROM clean_order_items
WHERE price >= 500.00
ORDER BY price DESC
LIMIT 5;
"""

pd.read_sql_query(q2, conn)

,order_id,order_item_id,product_id,seller_id,price,freight_value
0,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,6735.0,194.31
1,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,6729.0,193.21
2,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,6499.0,227.66
3,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,4799.0,151.34
4,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,4690.0,74.34


In [7]:
# Query 3: Target High-Freight Shipping Costs
q3 = """
SELECT 
    order_id,
    product_id,
    price,
    freight_value
FROM clean_order_items
WHERE freight_value > 100.00
ORDER BY freight_value DESC
LIMIT 5;
"""

pd.read_sql_query(q3, conn)

,order_id,product_id,price,freight_value
0,a77e1550db865202c56b19ddc6dc4d53,ec31d2a17b299511e7c8627be9337b9b,979.00,409.68
1,076d1555fb53a89b0ef4d529e527a0f6,a3cd9517ebf5a50dca25acce54f3b171,2338.08,375.28
2,3fde74c28a3d5d618c00f26d51baafa0,a3cd9517ebf5a50dca25acce54f3b171,2338.08,375.28
3,9f49bd16053df810384e793386312674,256a9c364b75753b97bee410c9491ad8,1149.00,339.59
4,264a7e199467906c0727394df82d1a6a,97c948ebc8c04b26b7bbb095d4228f2a,1050.00,338.30


In [8]:
# Query 4: Unfavorable Customer Review Ratings
q4 = """
SELECT 
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message
FROM clean_reviews
WHERE review_score <= 2
ORDER BY review_score ASC
LIMIT 5;
"""

pd.read_sql_query(q4, conn)

,review_id,order_id,review_score,review_comment_title,review_comment_message
0,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,No Title,No Comment Provided
1,373cbeecea8286a2b66c97b1b157ec46,583174fbe37d3d5f0d6661be3aad1786,1,Não chegou meu produto,Péssimo
2,2c5e27fc178bde7ac173c9c62c31b070,0ce9a24111d850192a933fcaab6fbad3,1,No Title,Não gostei ! Comprei gato por lebre
3,58044bca115705a48fe0e00a21390c54,68e55ca79d04a79f20d4bfc0146f4b66,1,No Title,Sempre compro pela Internet e a entrega ocorre...
4,9fd59cd04b42f600df9f25e54082a8d1,3c314f50bc654f3c4e317b055681dff9,1,No Title,Nada de chegar o meu pedido.


In [9]:
# Query 5: Multi-Installment Credit Purchases
q5 = """
SELECT 
    order_id,
    payment_type,
    payment_installments,
    payment_value
FROM clean_payments
WHERE payment_type = 'credit_card' 
  AND payment_installments >= 10
ORDER BY payment_value DESC
LIMIT 5;
"""

pd.read_sql_query(q5, conn)

,order_id,payment_type,payment_installments,payment_value
0,a96610ab360d42a2e5335a3998b4718a,credit_card,10,4950.34
1,426a9742b533fc6fed17d1fd6d143d7e,credit_card,10,4513.32
2,68101694e5c5dc7330c91e1bbc36214f,credit_card,10,4175.26
3,9de73f3e6157169ad6c32b9f313c7dcb,credit_card,10,3899.00
4,a53e05ecd2ed1f46a2b8e1f5828be7c6,credit_card,10,3826.80


In [10]:
# Query 6: Late Carrier Dispatch Dates
q6 = """
SELECT 
    order_id,
    order_purchase_timestamp,
    order_delivered_carrier_date,
    order_estimated_delivery_date
FROM clean_orders
WHERE order_delivered_carrier_date > order_estimated_delivery_date
  AND order_status = 'delivered'
LIMIT 5;
"""

pd.read_sql_query(q6, conn)

,order_id,order_purchase_timestamp,order_delivered_carrier_date,order_estimated_delivery_date
0,203096f03d82e0dffbc41ebc2e2bcfb7,2017-09-18 14:31:30,2017-10-06 17:50:03,2017-09-28
1,a5474c0071dd5d1074e12d417078bbd0,2018-07-30 22:41:44,2018-08-02 10:35:00,2018-08-02
2,234c056c50619f48da64f731c48242b4,2018-08-14 14:49:15,2018-08-31 15:25:00,2018-08-23
3,4190ab61a7fced69f3ee84d1da1120cc,2017-12-08 11:38:00,2018-01-12 00:35:33,2018-01-11
4,a06c43ed81f5c604287461f4d21949ce,2017-10-24 17:39:10,2017-11-20 21:15:00,2017-11-17


In [11]:
# Query 7: High-Volume Customer States
q7 = """
SELECT 
    customer_id,
    customer_unique_id,
    customer_city,
    customer_state
FROM clean_customers
WHERE customer_state IN ('SP', 'RJ', 'MG')
ORDER BY customer_state, customer_city
LIMIT 5;
"""

pd.read_sql_query(q7, conn)

,customer_id,customer_unique_id,customer_city,customer_state
0,f11eb8f0b8b87510a93e3e1aa10b0ade,64ee476500a01beb94df40f97a108c50,abadia dos dourados,MG
1,a23e3f9a2b656b23b7e52075964b42cd,afddf43a03a9941624ed42c0b2c17280,abadia dos dourados,MG
2,9e01f714a2b3b8962c222cf2b74c20dc,e1feae9083c4c2895ddf6dc80526a85d,abadia dos dourados,MG
3,5e9e1ae42e02df93e9a591e86fd531a3,28af9604f7830ef6d1230fb575c39eb1,abaete,MG
4,ff0d62f8be4c098e6306f39bc6ebded4,ab26557b289641b505abd795c0913683,abaete,MG


In [12]:
# Query 8: Revenue and AOV by Category
q8 = """
SELECT 
    p.product_category_name_english,
    COUNT(i.order_id) AS total_items_sold,
    ROUND(SUM(i.price), 2) AS gross_revenue_brl,
    ROUND(AVG(i.price), 2) AS avg_item_price_brl
FROM clean_order_items i
JOIN clean_products p ON i.product_id = p.product_id
GROUP BY p.product_category_name_english
ORDER BY gross_revenue_brl DESC
LIMIT 5;
"""

pd.read_sql_query(q8, conn)

,product_category_name_english,total_items_sold,gross_revenue_brl,avg_item_price_brl
0,health_beauty,9670,1258681.34,130.16
1,watches_gifts,5991,1205005.68,201.14
2,bed_bath_table,11115,1036988.68,93.30
3,sports_leisure,8641,988048.97,114.34
4,computers_accessories,7827,911954.32,116.51


In [13]:
# Query 9: Order Status Volume Breakdown
q9 = """
SELECT 
    order_status,
    COUNT(order_id) AS order_count,
    ROUND(COUNT(order_id) * 100.0 / (SELECT COUNT(*) FROM clean_orders), 2) AS percentage_of_total
FROM clean_orders
GROUP BY order_status
ORDER BY order_count DESC;
"""

pd.read_sql_query(q9, conn)

,order_status,order_count,percentage_of_total
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [14]:
# Query 10: Payment Type Distribution
q10 = """
SELECT 
    payment_type,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(payment_value), 2) AS total_processed_value,
    ROUND(AVG(payment_value), 2) AS avg_payment_value
FROM clean_payments
GROUP BY payment_type
ORDER BY total_processed_value DESC;
"""

pd.read_sql_query(q10, conn)

,payment_type,order_count,total_processed_value,avg_payment_value
0,credit_card,76505,12542084.19,163.32
1,boleto,19784,2869361.27,145.03
2,voucher,3866,379436.87,65.70
3,debit_card,1528,217989.79,142.57
4,not_defined,3,0.00,0.00


In [15]:
# Query 11: High-Volume Categories
q11 = """
SELECT 
    p.product_category_name_english,
    COUNT(i.order_id) AS total_sales_count,
    ROUND(SUM(i.price), 2) AS total_category_revenue
FROM clean_order_items i
JOIN clean_products p ON i.product_id = p.product_id
GROUP BY p.product_category_name_english
HAVING total_category_revenue > 100000.00
ORDER BY total_category_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(q11, conn)

,product_category_name_english,total_sales_count,total_category_revenue
0,health_beauty,9670,1258681.34
1,watches_gifts,5991,1205005.68
2,bed_bath_table,11115,1036988.68
3,sports_leisure,8641,988048.97
4,computers_accessories,7827,911954.32
5,furniture_decor,8334,729762.49
6,cool_stuff,3796,635290.85
7,housewares,6964,632248.66
8,auto,4235,592720.11
9,garden_tools,4347,485256.46


In [16]:
# Query 12: Price Tiers with CASE
q12 = """
SELECT 
    order_id,
    price,
    CASE 
        WHEN price < 50.00 THEN 'Low Value (< $50)'
        WHEN price BETWEEN 50.00 AND 200.00 THEN 'Mid Value ($50 - $200)'
        ELSE 'High Value (> $200)'
    END AS price_tier
FROM clean_order_items
LIMIT 10;
"""

pd.read_sql_query(q12, conn)

,order_id,price,price_tier
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,Mid Value ($50 - $200)
1,00018f77f2f0320c557190d7a144bdd3,239.90,High Value (> $200)
2,000229ec398224ef6ca0657da4fc703e,199.00,Mid Value ($50 - $200)
3,00024acbcdf0a6daa1e931b038114c75,12.99,Low Value (< $50)
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,Mid Value ($50 - $200)
5,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,Low Value (< $50)
6,00054e8431b9d7675808bcb819fb4a32,19.90,Low Value (< $50)
7,000576fe39319847cbb9d288c5617fa6,810.00,High Value (> $200)
8,0005a1a1728c9d785b8e2b08b904576c,145.95,Mid Value ($50 - $200)
9,0005f50442cb953dcd1d21e1fb923495,53.99,Mid Value ($50 - $200)


In [17]:
# Query 13: SLA Met vs SLA Breached
q13 = """
SELECT 
    CASE 
        WHEN is_delayed = 1 THEN 'SLA Breached (Late)'
        ELSE 'SLA Met (On-Time / Early)'
    END AS delivery_performance,
    COUNT(order_id) AS order_count,
    ROUND(AVG(delivery_time_days), 2) AS avg_fulfillment_days
FROM clean_orders
WHERE order_status = 'delivered'
GROUP BY is_delayed;
"""

pd.read_sql_query(q13, conn)

,delivery_performance,order_count,avg_fulfillment_days
0,SLA Met (On-Time / Early),88652,10.88
1,SLA Breached (Late),7826,31.52


In [18]:
# Query 14: High Delay Rate States (> 10%)
q14 = """
SELECT 
    c.customer_state,
    COUNT(o.order_id) AS total_orders,
    SUM(CASE WHEN o.is_delayed = 1 THEN 1 ELSE 0 END) AS delayed_orders,
    ROUND(SUM(CASE WHEN o.is_delayed = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(o.order_id), 2) AS late_rate_pct
FROM clean_orders o
JOIN clean_customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
HAVING late_rate_pct > 10.00
ORDER BY late_rate_pct DESC;
"""

pd.read_sql_query(q14, conn)

,customer_state,total_orders,delayed_orders,late_rate_pct
0,AL,397,95,23.93
1,MA,717,141,19.67
2,PI,476,76,15.97
3,CE,1279,196,15.32
4,SE,335,51,15.22
5,BA,3256,457,14.04
6,RJ,12350,1664,13.47
7,TO,274,35,12.77
8,PA,946,117,12.37
9,ES,1995,244,12.23


In [19]:
# Query 15: Multi-table Order Line-Item Header
q15 = """
SELECT 
    o.order_id,
    o.order_purchase_timestamp,
    c.customer_state,
    p.product_category_name_english,
    i.price,
    i.freight_value
FROM clean_orders o
INNER JOIN clean_customers c ON o.customer_id = c.customer_id
INNER JOIN clean_order_items i ON o.order_id = i.order_id
INNER JOIN clean_products p ON i.product_id = p.product_id
WHERE o.order_status = 'delivered'
LIMIT 10;
"""

pd.read_sql_query(q15, conn)

,order_id,order_purchase_timestamp,customer_state,product_category_name_english,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,SP,housewares,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,BA,perfumery,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,GO,auto,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,RN,pet_shop,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,SP,stationery,19.90,8.72
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:05,PR,auto,147.90,27.36
6,6514b8ad8028c9f2cc2374ded245783f,2017-05-16 13:10:30,RJ,auto,59.99,15.17
7,76c6e866289321a7c93b82b54852dc33,2017-01-23 18:29:09,RS,furniture_decor,19.90,16.05
8,e69bfb5eb88e0ed6a785585b27e16dbf,2017-07-29 11:55:02,SP,office_furniture,149.99,19.77
9,e6ce16cb79ec1d90b1da9085a6118aeb,2017-05-16 19:41:10,RJ,garden_tools,99.00,30.53


In [20]:
# Query 16: Lowest Rated Active Sellers
q16 = """
SELECT 
    s.seller_id,
    s.seller_state,
    COUNT(DISTINCT r.review_id) AS total_reviews,
    ROUND(AVG(r.review_score), 2) AS avg_seller_rating
FROM clean_sellers s
JOIN clean_order_items i ON s.seller_id = i.seller_id
JOIN clean_reviews r ON i.order_id = r.order_id
GROUP BY s.seller_id, s.seller_state
HAVING total_reviews >= 20
ORDER BY avg_seller_rating ASC
LIMIT 10;
"""

pd.read_sql_query(q16, conn)

,seller_id,seller_state,total_reviews,avg_seller_rating
0,ffff564a4f9085cd26170f4732393726,SP,20,2.10
1,1ca7077d890b907f89be8c954a02686a,SP,114,2.20
2,2709af9587499e95e803a6498a5a56e9,SP,25,2.57
3,2eb70248d66e0e3ef83659f71b244378,SP,199,2.71
4,b19f3ca2ea475913750f25a5c37c8d8f,MG,24,2.79
5,602044f2c16190c2c6e45eb35c2e21cb,SP,51,2.93
6,54965bbe3e4f07ae045b90b0b8541f52,PR,74,2.94
7,a49928bcdf77c55c6d6e05e09a9b4ca5,SP,98,2.95
8,972d0f9cf61b499a4812cf0bfa3ad3c4,SC,79,2.96
9,2a1348e9addc1af5aaa619b1a3679d6b,MG,51,3.00


In [21]:
# Query 17: Lead Conversion Rate by Channel
q17 = """
SELECT 
    m.origin AS acquisition_channel,
    COUNT(DISTINCT m.mql_id) AS total_leads,
    COUNT(DISTINCT c.seller_id) AS converted_sellers,
    ROUND(COUNT(DISTINCT c.seller_id) * 100.0 / COUNT(DISTINCT m.mql_id), 2) AS conversion_rate_pct
FROM olist_marketing_qualified_leads_dataset m
LEFT JOIN olist_closed_deals_dataset c ON m.mql_id = c.mql_id
GROUP BY m.origin
ORDER BY total_leads DESC;
"""

pd.read_sql_query(q17, conn)

,acquisition_channel,total_leads,converted_sellers,conversion_rate_pct
0,organic_search,2296,271,11.80
1,paid_search,1586,195,12.30
2,social,1350,75,5.56
3,unknown,1099,179,16.29
4,direct_traffic,499,56,11.22
5,email,493,15,3.04
6,referral,284,24,8.45
7,other,150,4,2.67
8,display,118,6,5.08
9,other_publicities,65,3,4.62


In [22]:
# Query 18: Origin-Destination Freight and Delivery Days
q18 = """
SELECT 
    s.seller_state AS origin_state,
    c.customer_state AS destination_state,
    COUNT(DISTINCT o.order_id) AS order_volume,
    ROUND(AVG(o.delivery_time_days), 2) AS avg_transit_days,
    ROUND(AVG(i.freight_value), 2) AS avg_freight_cost
FROM clean_orders o
JOIN clean_order_items i ON o.order_id = i.order_id
JOIN clean_sellers s ON i.seller_id = s.seller_id
JOIN clean_customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_state, c.customer_state
ORDER BY order_volume DESC
LIMIT 10;
"""

pd.read_sql_query(q18, conn)

,origin_state,destination_state,order_volume,avg_transit_days,avg_freight_cost
0,SP,SP,30834,7.93,13.20
1,SP,RJ,8188,16.11,20.39
2,SP,MG,7469,12.29,20.25
3,SP,RS,3614,16.00,20.61
4,SP,PR,3130,12.77,20.44
5,PR,SP,2959,11.12,19.21
6,MG,SP,2580,10.91,21.35
7,SP,SC,2336,15.83,20.81
8,SP,BA,2313,19.73,24.32
9,MG,MG,1539,8.63,17.13


In [23]:
# Query 19: Unmapped Catalog Items
q19 = """
SELECT 
    p.product_id,
    p.product_category_name_english
FROM clean_products p
WHERE p.product_category_name_english = 'unknown' OR p.product_category_name_english IS NULL
LIMIT 10;
"""

pd.read_sql_query(q19, conn)

,product_id,product_category_name_english
0,a41e356c76fab66334f36de622ecbd3a,unknown
1,d8dee61c2034d6d075997acef1870e9b,unknown
2,56139431d72cd51f19eb9f7dae4d1617,unknown
3,46b48281eb6d663ced748f324108c733,unknown
4,5fb61f482620cb672f5e586bb132eae9,unknown
5,e10758160da97891c2fdcbc35f0f031d,unknown
6,39e3b9b12cd0bf8ee681bbc1c130feb5,unknown
7,794de06c32a626a5692ff50e4985d36f,unknown
8,7af3e2da474486a3519b0cba9dea8ad9,unknown
9,629beb8e7317703dcc5f35b5463fd20e,unknown


In [24]:
# Query 20: Orders Higher Than Average Spend
q20 = """
SELECT 
    order_id,
    SUM(price) AS order_total_price
FROM clean_order_items
GROUP BY order_id
HAVING order_total_price > (
    SELECT AVG(item_total) 
    FROM (
        SELECT SUM(price) AS item_total 
        FROM clean_order_items 
        GROUP BY order_id
    ) AS inner_order_totals
)
ORDER BY order_total_price DESC
LIMIT 10;
"""

pd.read_sql_query(q20, conn)

,order_id,order_total_price
0,03caa2c082116e1d31e67e9ae3700499,13440.0
1,736e1922ae60d0d6a89247b851902527,7160.0
2,0812eb902a67711a1cb742b3cdaa65ae,6735.0
3,fefacc66af859508bf1a7934eab1e97f,6729.0
4,f5136e38d1a14a4dbd87dff67da82701,6499.0
5,2cc9089445046817a7539d90805e6e5a,5934.6
6,a96610ab360d42a2e5335a3998b4718a,4799.0
7,199af31afc78c699f0dbf71fb178d4d4,4690.0
8,b4c4b76c642808cbe472a32b86cddc95,4599.9
9,8dbc85d1447242f3b127dda390d56e19,4590.0


In [25]:
# Query 21: Sellers in High-Delay States
q21 = """
SELECT 
    s.seller_id,
    s.seller_state
FROM clean_sellers s
WHERE s.seller_state IN (
    SELECT c.customer_state
    FROM clean_orders o
    JOIN clean_customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
    HAVING AVG(CASE WHEN o.is_delayed = 1 THEN 1.0 ELSE 0.0 END) > 0.12
)
LIMIT 10;
"""

pd.read_sql_query(q21, conn)

,seller_id,seller_state
0,ce3ad9de960102d0677a81f5d0bb7b2d,RJ
1,c240c4061717ac1806ae6ee72be3533b,RJ
2,1444c08e64d55fb3c25f0f09c07ffcf2,BA
3,d2e753bb80b7d4faa77483ed00edc8ca,BA
4,0747d5bb69f0586cc869d8af4c50f93e,RJ
5,c89cf7c468a48af70aada384e722f9e2,RJ
6,abe42c5d03695b4257b5c6cbf4e6784e,RJ
7,79b93a308a97792cf53ac75f46da00b5,RJ
8,63ffcb71394dd8ea3872ed9ffda17c74,RJ
9,9c4d31c7e46ab03a43fc06e3142afd4e,RJ


In [26]:
# Query 22: Repeat Customers
q22 = """
SELECT 
    customer_unique_id,
    COUNT(customer_id) AS total_orders_placed
FROM clean_customers
GROUP BY customer_unique_id
HAVING total_orders_placed > 1
ORDER BY total_orders_placed DESC
LIMIT 10;
"""

pd.read_sql_query(q22, conn)

,customer_unique_id,total_orders_placed
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,de34b16117594161a6a89c50b289d35a,6
7,dc813062e0fc23409cd255f7f53c7074,6
8,63cfc61cee11cbe306bff5857d00bfe4,6
9,47c1a3033b8b77b3ab6e109eb4d5fdf3,6


In [27]:
# Query 23: Exceptionally High Freight Items
q23 = """
SELECT 
    i.order_id,
    i.product_id,
    i.freight_value
FROM clean_order_items i
WHERE i.freight_value > (
    SELECT AVG(freight_value) * 2.5 FROM clean_order_items
)
ORDER BY i.freight_value DESC
LIMIT 10;
"""

pd.read_sql_query(q23, conn)

,order_id,product_id,freight_value
0,a77e1550db865202c56b19ddc6dc4d53,ec31d2a17b299511e7c8627be9337b9b,409.68
1,076d1555fb53a89b0ef4d529e527a0f6,a3cd9517ebf5a50dca25acce54f3b171,375.28
2,3fde74c28a3d5d618c00f26d51baafa0,a3cd9517ebf5a50dca25acce54f3b171,375.28
3,9f49bd16053df810384e793386312674,256a9c364b75753b97bee410c9491ad8,339.59
4,264a7e199467906c0727394df82d1a6a,97c948ebc8c04b26b7bbb095d4228f2a,338.30
5,c7a07ddd52bbe18b61da49a8d89853d3,97c948ebc8c04b26b7bbb095d4228f2a,322.10
6,0b6230647ed16f4b3e70282dc4b5b87f,46e24ce614899e36617e37ea1e4aa6ff,321.88
7,0822bcde10bb5d023755a71bc8f7797f,363a9f5b97bf194da23858be722a7aa5,321.46
8,43bdbd9dc0931d72befdf4765af6c442,7e53e051875b2a0c9f22acd8a9a29a20,317.47
9,6ddfbf514959b49b6410c01ad93054bb,363a9f5b97bf194da23858be722a7aa5,314.40


In [28]:
# Query 24: Customer Lifetime Value CTE
q24 = """
WITH CustomerMetrics AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(i.price), 2) AS total_monetary_spend,
        MIN(o.order_purchase_timestamp) AS first_purchase_date,
        MAX(o.order_purchase_timestamp) AS last_purchase_date
    FROM clean_customers c
    JOIN clean_orders o ON c.customer_id = o.customer_id
    JOIN clean_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT 
    customer_unique_id,
    total_orders,
    total_monetary_spend,
    ROUND(total_monetary_spend / total_orders, 2) AS avg_order_value
FROM CustomerMetrics
ORDER BY total_monetary_spend DESC
LIMIT 10;
"""

pd.read_sql_query(q24, conn)

,customer_unique_id,total_orders,total_monetary_spend,avg_order_value
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.0,13440.0
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.0,3694.0
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.0,7160.0
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.0,6735.0
4,459bef486812aa25204be022145caa62,1,6729.0,6729.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.0,6499.0
6,4007669dec559734d6f53e029e360987,1,5934.6,5934.6
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.0,4690.0
8,48e1ac109decbb87765a3eade6854098,1,4590.0,4590.0
9,a229eba70ec1c2abef51f04987deb7a5,1,4400.0,4400.0


In [29]:
# Query 25: Seller Revenue & Ratings Multi-CTE
q25 = """
WITH SellerSales AS (
    SELECT 
        seller_id,
        COUNT(DISTINCT order_id) AS completed_orders,
        ROUND(SUM(price), 2) AS total_gross_revenue
    FROM clean_order_items
    GROUP BY seller_id
),
SellerRatings AS (
    SELECT 
        i.seller_id,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM clean_order_items i
    JOIN clean_reviews r ON i.order_id = r.order_id
    GROUP BY i.seller_id
)
SELECT 
    s.seller_id,
    s.completed_orders,
    s.total_gross_revenue,
    r.avg_review_score
FROM SellerSales s
JOIN SellerRatings r ON s.seller_id = r.seller_id
WHERE s.completed_orders >= 10
ORDER BY s.total_gross_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(q25, conn)

,seller_id,completed_orders,total_gross_revenue,avg_review_score
0,4869f7a5dfa277a7dca6462dcf3b52b2,1132,229472.63,4.12
1,53243585a1d6dc2643021fd1853d8905,358,222776.05,4.08
2,4a3ca9315b744ce9f8e9374361493884,1806,200472.92,3.80
3,fa1c13f2614d7b5c4749cbc52fecda94,585,194042.03,4.34
4,7c67e1448b00f6e969d365cea6b010ab,982,187923.89,3.35
5,7e93a43ef30c4f03f38b393420bc753a,336,176431.87,4.21
6,da8622b14eb17ae2831f4ac5b9dab84a,1314,160236.57,4.07
7,7a67c85e85bb2ce8582c35f2203ad736,1160,141745.53,4.23
8,1025f0e2d44d7041d6cf58b6550e0bfa,915,138968.55,3.85
9,955fee9216a65b617aa5c0531780ce60,1287,135171.70,4.05


In [30]:
# Query 26: Regional Delivery Risk CTE
q26 = """
WITH StateLogistics AS (
    SELECT 
        c.customer_state,
        COUNT(o.order_id) AS total_orders,
        SUM(CASE WHEN o.is_delayed = 1 THEN 1 ELSE 0 END) AS delayed_orders,
        AVG(o.delivery_time_days) AS avg_delivery_days
    FROM clean_orders o
    JOIN clean_customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
)
SELECT 
    customer_state,
    total_orders,
    delayed_orders,
    ROUND((delayed_orders * 100.0 / total_orders), 2) AS delay_rate_pct,
    ROUND(avg_delivery_days, 1) AS avg_delivery_days
FROM StateLogistics
ORDER BY delay_rate_pct DESC
LIMIT 10;
"""

pd.read_sql_query(q26, conn)

,customer_state,total_orders,delayed_orders,delay_rate_pct,avg_delivery_days
0,AL,397,95,23.93,24.5
1,MA,717,141,19.67,21.6
2,PI,476,76,15.97,19.5
3,CE,1279,196,15.32,21.3
4,SE,335,51,15.22,21.5
5,BA,3256,457,14.04,19.3
6,RJ,12350,1664,13.47,15.3
7,TO,274,35,12.77,17.7
8,PA,946,117,12.37,23.8
9,ES,1995,244,12.23,15.8


In [31]:
# Query 27: Sales Onboarding Closing Velocity CTE
q27 = """
WITH SalesFunnel AS (
    SELECT 
        c.sr_id,
        COUNT(c.mql_id) AS deals_won,
        AVG(julianday(c.won_date) - julianday(m.first_contact_date)) AS avg_closing_days
    FROM olist_closed_deals_dataset c
    JOIN olist_marketing_qualified_leads_dataset m ON c.mql_id = m.mql_id
    GROUP BY c.sr_id
)
SELECT 
    sr_id,
    deals_won,
    ROUND(avg_closing_days, 1) AS avg_closing_days
FROM SalesFunnel
ORDER BY deals_won DESC
LIMIT 10;
"""

pd.read_sql_query(q27, conn)

,sr_id,deals_won,avg_closing_days
0,4ef15afb4b2723d8f3d81e51ec7afefe,133,31.2
1,d3d1e91a157ea7f90548eef82f1955e3,82,47.3
2,6565aa9ce3178a5caf6171827af3a9ba,74,26.5
3,85fc447d336637ba1df43e793199fbc8,64,34.4
4,495d4e95a8cf8bbf8b432b612a2aa328,63,49.2
5,2695de1affa7750089c0455f8ce27021,59,49.0
6,fbf4aef3f6915dc0c3c97d6812522f6a,59,22.6
7,de63de0d10a6012430098db33c679b0b,53,62.4
8,9ae085775a198122c5586fa830ff7f2b,51,42.3
9,c638112b43f1d1b86dcabb0da720c901,36,23.0


In [32]:
# Query 28: Product Revenue Rank in Category
q28 = """
SELECT 
    p.product_category_name_english,
    i.product_id,
    ROUND(SUM(i.price), 2) AS product_revenue,
    DENSE_RANK() OVER (
        PARTITION BY p.product_category_name_english 
        ORDER BY SUM(i.price) DESC
    ) AS category_revenue_rank
FROM clean_order_items i
JOIN clean_products p ON i.product_id = p.product_id
GROUP BY p.product_category_name_english, i.product_id
LIMIT 10;
"""

pd.read_sql_query(q28, conn)

,product_category_name_english,product_id,product_revenue,category_revenue_rank
0,agro_industry_and_commerce,11250b0d4b709fee92441c5f34122aed,9111.0,1
1,agro_industry_and_commerce,423a6644f0aa529e8828ff1f91003690,8043.0,2
2,agro_industry_and_commerce,672e757f331900b9deea127a2a7b79fd,6885.0,3
3,agro_industry_and_commerce,c183fd5d2abf05873fa6e1014ed9e06c,5934.6,4
4,agro_industry_and_commerce,2b69866f22de8dad69c976771daba91c,2990.0,5
5,agro_industry_and_commerce,c89226b8a795ae3d6bca9d90b20dbf04,2821.5,6
6,agro_industry_and_commerce,5fb0955cb683eb6f65a1f613e502eef5,2720.0,7
7,agro_industry_and_commerce,b7a60a397d4efd05c1b5d398fb9f9097,2399.0,8
8,agro_industry_and_commerce,cd5df6a3db7a3d064a55afd08289d762,2360.0,9
9,agro_industry_and_commerce,cd2f5c10e4e8dbc701f0bb68a09fdfe8,2199.0,10


In [33]:
# Query 29: Month-over-Month Revenue Growth
q29 = """
WITH MonthlyRevenue AS (
    SELECT 
        purchase_month,
        ROUND(SUM(i.price), 2) AS current_month_revenue
    FROM clean_orders o
    JOIN clean_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY purchase_month
)
SELECT 
    purchase_month,
    current_month_revenue,
    LAG(current_month_revenue, 1) OVER (ORDER BY purchase_month) AS previous_month_revenue,
    ROUND(
        (current_month_revenue - LAG(current_month_revenue, 1) OVER (ORDER BY purchase_month)) * 100.0 /
        LAG(current_month_revenue, 1) OVER (ORDER BY purchase_month), 
    2) AS mom_growth_pct
FROM MonthlyRevenue
LIMIT 10;
"""

pd.read_sql_query(q29, conn)

,purchase_month,current_month_revenue,previous_month_revenue,mom_growth_pct
0,2016-09,134.97,NaN,NaN
1,2016-10,40325.11,134.97,29777.09
2,2016-12,10.90,40325.11,-99.97
3,2017-01,111798.36,10.90,1025573.03
4,2017-02,234223.40,111798.36,109.51
5,2017-03,359198.85,234223.40,53.36
6,2017-04,340669.68,359198.85,-5.16
7,2017-05,489338.25,340669.68,43.64
8,2017-06,421923.37,489338.25,-13.78
9,2017-07,481604.52,421923.37,14.15


In [34]:
# Query 30: Running Cumulative Revenue
q30 = """
WITH DailySales AS (
    SELECT 
        DATE(order_purchase_timestamp) AS sale_date,
        ROUND(SUM(i.price), 2) AS daily_revenue
    FROM clean_orders o
    JOIN clean_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY DATE(order_purchase_timestamp)
)
SELECT 
    sale_date,
    daily_revenue,
    SUM(daily_revenue) OVER (ORDER BY sale_date) AS cumulative_running_revenue
FROM DailySales
LIMIT 10;
"""

pd.read_sql_query(q30, conn)

,sale_date,daily_revenue,cumulative_running_revenue
0,2016-09-15,134.97,134.97
1,2016-10-03,441.98,576.95
2,2016-10-04,8595.89,9172.84
3,2016-10-05,6169.77,15342.61
4,2016-10-06,5889.96,21232.57
5,2016-10-07,6075.35,27307.92
6,2016-10-08,7592.89,34900.81
7,2016-10-09,2399.70,37300.51
8,2016-10-10,3159.57,40460.08
9,2016-12-23,10.90,40470.98


In [35]:
# Query 31: Customer Order Sequence Numbering
q31 = """
SELECT 
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    ROW_NUMBER() OVER (
        PARTITION BY c.customer_unique_id 
        ORDER BY o.order_purchase_timestamp ASC
    ) AS customer_order_sequence
FROM clean_orders o
JOIN clean_customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
LIMIT 10;
"""

pd.read_sql_query(q31, conn)

,customer_unique_id,order_id,order_purchase_timestamp,customer_order_sequence
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27,1
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03,1
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41,1
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42,1
5,0004bd2a26a76fe21f786e4fbd80607f,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16,1
6,00050ab1314c0e55a6ca13cf7181fecf,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23,1
7,00053a61a98854899e70ed204dd4bafe,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41,1
8,0005e1862207bf6ccc02e4228effd9a0,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12,1


In [36]:
# Query 32: Top 3 Sellers Per State
q32 = """
WITH RankedSellers AS (
    SELECT 
        s.seller_state,
        s.seller_id,
        ROUND(SUM(i.price), 2) AS total_seller_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY s.seller_state 
            ORDER BY SUM(i.price) DESC
        ) AS state_rank
    FROM clean_sellers s
    JOIN clean_order_items i ON s.seller_id = i.seller_id
    GROUP BY s.seller_state, s.seller_id
)
SELECT 
    seller_state,
    seller_id,
    total_seller_revenue,
    state_rank
FROM RankedSellers
WHERE state_rank <= 3
ORDER BY seller_state, state_rank
LIMIT 10;
"""

pd.read_sql_query(q32, conn)

,seller_state,seller_id,total_seller_revenue,state_rank
0,AC,4be2e7f96b4fd749d52dff41f80e39dd,267.00,1
1,AM,327b89b872c14d1c0be7235ef4871685,1177.00,1
2,BA,53243585a1d6dc2643021fd1853d8905,222776.05,1
3,BA,c72de06d72748d1a0dfb2125be43ba63,17522.00,2
4,BA,75d34ebb1bd0bd7dde40dd507b8169c3,15048.28,3
5,CE,bbf9ad41dca6603e614efcdad7aab8c4,7846.00,1
6,CE,dbdd0ec73a4817971d962698f2fea022,6384.00,2
7,CE,8d79c8a04e42d722a75097ce5cbcf2ef,2777.72,3
8,DF,44073f8b7e41514de3b7815dd0237f4f,18729.64,1
9,DF,f3b80352b986ab4d1057a4b724be19d0,10505.10,2


In [37]:
# Query 33: Days Between Customer Repeat Orders
q33 = """
WITH CustomerOrders AS (
    SELECT 
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        LEAD(o.order_purchase_timestamp, 1) OVER (
            PARTITION BY c.customer_unique_id 
            ORDER BY o.order_purchase_timestamp ASC
        ) AS next_order_timestamp
    FROM clean_orders o
    JOIN clean_customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
)
SELECT 
    customer_unique_id,
    order_id,
    order_purchase_timestamp,
    next_order_timestamp,
    ROUND(julianday(next_order_timestamp) - julianday(order_purchase_timestamp), 1) AS days_until_next_order
FROM CustomerOrders
WHERE next_order_timestamp IS NOT NULL
LIMIT 10;
"""

pd.read_sql_query(q33, conn)

,customer_unique_id,order_id,order_purchase_timestamp,next_order_timestamp,days_until_next_order
0,004288347e5e88a27ded2bb23747066c,a61d617fbe5bd006e40d3a0988fc844b,2017-07-27 14:13:03,2018-01-14 07:36:54,170.7
1,00a39521eb40f7012db50455bf083460,7d32c87acba91ed87ebd98310fe1c54d,2018-05-23 20:14:21,2018-06-03 10:12:57,10.6
2,00cc12a6d8b578b8ebd21ea4e2ae8b27,64307ceb91666760cf3ff463618302fd,2017-03-21 19:25:22,2017-03-21 19:25:23,0.0
3,011575986092c30523ecb71ff10cb473,0f28d51fdd4828907bdc58b57f672e84,2018-02-17 15:54:49,2018-04-18 21:58:08,60.3
4,011b4adcd54683b480c4d841250a987f,f05a68aaa0d8f89e758c7134d53fa22a,2017-08-22 12:51:29,2018-02-15 11:40:57,177.0
5,012452d40dafae4df401bced74cdb490,ce2b4f2836d78829e4796213d536101e,2017-06-18 22:46:42,2018-05-14 12:12:45,329.6
6,012a218df8995d3ec3bb221828360c86,e89e48d863049f7ca8a49758ae99f58a,2018-05-07 10:28:17,2018-06-18 13:08:38,42.1
7,013ef03e0f3f408dd9bf555e4edcdc0a,c94047ec162181b78322184f7a9aee43,2018-06-21 04:46:11,2018-07-20 04:13:54,29.0
8,013f4353d26bb05dc6652f1269458d8d,ddc5df3b1c497720bae1692c170baec0,2017-11-24 13:33:20,2017-11-28 13:30:58,4.0
9,015557c9912277312b9073947804a7ba,482c052df6392e43296eba90754938fc,2017-03-23 22:45:46,2017-05-01 14:48:33,38.7


In [38]:
# Query 34: 7-Day Moving Average Revenue
q34 = """
WITH DailyRevenue AS (
    SELECT 
        DATE(order_purchase_timestamp) AS sales_date,
        SUM(i.price) AS revenue
    FROM clean_orders o
    JOIN clean_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY DATE(order_purchase_timestamp)
)
SELECT 
    sales_date,
    ROUND(revenue, 2) AS daily_revenue,
    ROUND(AVG(revenue) OVER (
        ORDER BY sales_date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_7_day_revenue
FROM DailyRevenue
LIMIT 10;
"""

pd.read_sql_query(q34, conn)

,sales_date,daily_revenue,moving_avg_7_day_revenue
0,2016-09-15,134.97,134.97
1,2016-10-03,441.98,288.48
2,2016-10-04,8595.89,3057.61
3,2016-10-05,6169.77,3835.65
4,2016-10-06,5889.96,4246.51
5,2016-10-07,6075.35,4551.32
6,2016-10-08,7592.89,4985.83
7,2016-10-09,2399.70,5309.36
8,2016-10-10,3159.57,5697.59
9,2016-12-23,10.90,4471.16


In [39]:
# Query 35: Category Share of Total Revenue
q35 = """
WITH CategoryTotals AS (
    SELECT 
        p.product_category_name_english AS category,
        SUM(i.price) AS category_revenue
    FROM clean_order_items i
    JOIN clean_products p ON i.product_id = p.product_id
    GROUP BY p.product_category_name_english
)
SELECT 
    category,
    ROUND(category_revenue, 2) AS category_revenue_brl,
    ROUND(category_revenue * 100.0 / SUM(category_revenue) OVER(), 2) AS share_of_total_revenue_pct
FROM CategoryTotals
ORDER BY category_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(q35, conn)

,category,category_revenue_brl,share_of_total_revenue_pct
0,health_beauty,1258681.34,9.26
1,watches_gifts,1205005.68,8.87
2,bed_bath_table,1036988.68,7.63
3,sports_leisure,988048.97,7.27
4,computers_accessories,911954.32,6.71
5,furniture_decor,729762.49,5.37
6,cool_stuff,635290.85,4.67
7,housewares,632248.66,4.65
8,auto,592720.11,4.36
9,garden_tools,485256.46,3.57


In [40]:
# Query 36: Customer Spend Quartiles
q36 = """
WITH CustomerSpend AS (
    SELECT 
        c.customer_unique_id,
        SUM(i.price) AS total_spend
    FROM clean_customers c
    JOIN clean_orders o ON c.customer_id = o.customer_id
    JOIN clean_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT 
    customer_unique_id,
    ROUND(total_spend, 2) AS total_spend_brl,
    NTILE(4) OVER (ORDER BY total_spend DESC) AS spend_quartile
FROM CustomerSpend
LIMIT 10;
"""

pd.read_sql_query(q36, conn)

,customer_unique_id,total_spend_brl,spend_quartile
0,0a0a92112bd4c708ca5fde585afaa872,13440.0,1
1,da122df9eeddfedc1dc1f5349a1a690c,7388.0,1
2,763c8b1c9c68a0229c42c9fc6f662b93,7160.0,1
3,dc4802a71eae9be1dd28f5d788ceb526,6735.0,1
4,459bef486812aa25204be022145caa62,6729.0,1
5,ff4159b92c40ebe40454e3e6a7c35ed6,6499.0,1
6,4007669dec559734d6f53e029e360987,5934.6,1
7,eebb5dda148d3893cdaf5b5ca3040ccb,4690.0,1
8,48e1ac109decbb87765a3eade6854098,4590.0,1
9,a229eba70ec1c2abef51f04987deb7a5,4400.0,1


In [41]:
# Query 37: Price Variance vs Category Average
q37 = """
SELECT 
    i.order_id,
    p.product_category_name_english,
    i.price,
    ROUND(AVG(i.price) OVER (PARTITION BY p.product_category_name_english), 2) AS avg_category_price,
    ROUND(i.price - AVG(i.price) OVER (PARTITION BY p.product_category_name_english), 2) AS price_diff_from_avg
FROM clean_order_items i
JOIN clean_products p ON i.product_id = p.product_id
LIMIT 10;
"""

pd.read_sql_query(q37, conn)

,order_id,product_category_name_english,price,avg_category_price,price_diff_from_avg
0,00137e170939bba5a3134e2386413108,agro_industry_and_commerce,397.0,342.12,54.88
1,010b143d83a59b355cd5a75c0f0fd785,agro_industry_and_commerce,22.0,342.12,-320.12
2,010b143d83a59b355cd5a75c0f0fd785,agro_industry_and_commerce,22.0,342.12,-320.12
3,010b143d83a59b355cd5a75c0f0fd785,agro_industry_and_commerce,22.0,342.12,-320.12
4,010b143d83a59b355cd5a75c0f0fd785,agro_industry_and_commerce,22.0,342.12,-320.12
5,01af7745a283aeb0f0754495c4b51ac7,agro_industry_and_commerce,465.0,342.12,122.88
6,03211eda276999d5eaa8a167a4b6286a,agro_industry_and_commerce,479.0,342.12,136.88
7,04e00ba23c33890eaee39b02e8185cc2,agro_industry_and_commerce,33.9,342.12,-308.22
8,051d3d717e96b76b07fc9076b88d671f,agro_industry_and_commerce,58.5,342.12,-283.62
9,06c9818464bb97d2f13568dd66e904a9,agro_industry_and_commerce,549.0,342.12,206.88


In [42]:
# Query 38: Review Score Variance vs State Avg
q38 = """
SELECT 
    r.review_id,
    c.customer_state,
    r.review_score,
    ROUND(AVG(r.review_score) OVER (PARTITION BY c.customer_state), 2) AS state_avg_review_score,
    ROUND(r.review_score - AVG(r.review_score) OVER (PARTITION BY c.customer_state), 2) AS score_variance
FROM clean_reviews r
JOIN clean_orders o ON r.order_id = o.order_id
JOIN clean_customers c ON o.customer_id = c.customer_id
LIMIT 10;
"""

pd.read_sql_query(q38, conn)

,review_id,customer_state,review_score,state_avg_review_score,score_variance
0,2a3eba09af2fbcd3a20580e62e7b7465,AC,4,4.05,-0.05
1,a68b5b20f860f7904fd5486d3a2d5173,AC,3,4.05,-1.05
2,a2626526243e9646422678afc067063c,AC,5,4.05,0.95
3,e3a081060d6caba024c22dceb1f6398e,AC,5,4.05,0.95
4,05c524f633d97a11a12c94770d1243cb,AC,5,4.05,0.95
5,4a26c1010cecfd4fca2ebea67bb627d8,AC,4,4.05,-0.05
6,e4bdd5b6263b9fde2cce56ab02e91586,AC,2,4.05,-2.05
7,af73e2ba5098ca6c02e8079aa82bfa78,AC,2,4.05,-2.05
8,ec1788886b2f87ae29f8712b6e1b946d,AC,5,4.05,0.95
9,e832e5081ba346b67e56565f53bdd3a1,AC,5,4.05,0.95


In [43]:
# Query 39: Seller Revenue Percentile Rank
q39 = """
WITH SellerTotals AS (
    SELECT 
        seller_id,
        SUM(price) AS gross_revenue
    FROM clean_order_items
    GROUP BY seller_id
)
SELECT 
    seller_id,
    ROUND(gross_revenue, 2) AS gross_revenue_brl,
    ROUND(PERCENT_RANK() OVER (ORDER BY gross_revenue ASC) * 100, 2) AS seller_percentile_rank
FROM SellerTotals
LIMIT 10;
"""

pd.read_sql_query(q39, conn)

,seller_id,gross_revenue_brl,seller_percentile_rank
0,cf6f6bc4df3999b9c6440f124fb2f687,3.50,0.00
1,77128dec4bec4878c37ab7d6169d6f26,6.50,0.03
2,1fa2d3def6adfa70e58c276bb64fe5bb,6.90,0.06
3,702835e4b785b67a084280efca355756,7.60,0.10
4,34aefe746cd81b7f3b23253ea28bef39,8.00,0.13
5,ad14615bdd492b01b0d97922e87cb87f,8.25,0.16
6,4965a7002cca77301c82d3f91b82e1a9,8.49,0.19
7,0f94588695d71662beec8d883ffacf09,9.00,0.23
8,c18309219e789960add0b2255ca4b091,9.90,0.26
9,95cca791657aabeff15a07eb152d7841,9.99,0.29


In [44]:
# Query 40: Lead Sequence Position
q40 = """
SELECT 
    m.origin,
    m.mql_id,
    m.first_contact_date,
    c.won_date,
    CASE WHEN c.seller_id IS NOT NULL THEN 'Converted' ELSE 'Unconverted Lead' END AS conversion_status,
    ROW_NUMBER() OVER (PARTITION BY m.origin ORDER BY m.first_contact_date DESC) AS channel_lead_sequence
FROM olist_marketing_qualified_leads_dataset m
LEFT JOIN olist_closed_deals_dataset c ON m.mql_id = c.mql_id
LIMIT 10;
"""

pd.read_sql_query(q40, conn)

,origin,mql_id,first_contact_date,won_date,conversion_status,channel_lead_sequence
0,None,19f3ca179d09e8518d25a28d8e4e443f,2018-05-28,NaN,Unconverted Lead,1
1,None,99ba5c4097c6b8fef5ed774a1a6714b8,2018-05-26,NaN,Unconverted Lead,2
2,None,6b0ac5c5ab0f8939442c42704c5493b0,2018-05-25,NaN,Unconverted Lead,3
3,None,b51d0ceea431fc2d4569650b64bd22de,2018-05-21,NaN,Unconverted Lead,4
4,None,9e4903ea29841bc16eab71500292e9b0,2018-05-21,NaN,Unconverted Lead,5
5,None,8d57c03206c460f9e9637e46b8337713,2018-05-19,NaN,Unconverted Lead,6
6,None,4da6c4d07033d355453ce49273d591c6,2018-05-18,NaN,Unconverted Lead,7
7,None,d788c46ad6a6c020b5062c1a99f55b2c,2018-05-15,NaN,Unconverted Lead,8
8,None,d2ac71782272659e7171150d20d59158,2018-05-14,2018-05-15 21:17:04,Converted,9
9,None,280d8c73b56b79f1059539da36c7ae4a,2018-05-11,NaN,Unconverted Lead,10


In [45]:
# Query 41: Platform Executive Summary Metric Cross-Join
q41 = """
WITH OrderMetrics AS (
    SELECT 
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(CASE WHEN is_delayed = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(order_id) AS otdr_pct
    FROM clean_orders
    WHERE order_status = 'delivered'
),
RevenueMetrics AS (
    SELECT SUM(price) AS gross_revenue_brl, AVG(price) AS aov_brl FROM clean_order_items
),
FunnelMetrics AS (
    SELECT 
        COUNT(DISTINCT m.mql_id) AS total_leads,
        COUNT(DISTINCT c.seller_id) AS total_sellers_won,
        COUNT(DISTINCT c.seller_id) * 100.0 / COUNT(DISTINCT m.mql_id) AS conversion_rate_pct
    FROM olist_marketing_qualified_leads_dataset m
    LEFT JOIN olist_closed_deals_dataset c ON m.mql_id = c.mql_id
)
SELECT 
    o.total_orders,
    ROUND(r.gross_revenue_brl, 2) AS total_revenue_brl,
    ROUND(r.aov_brl, 2) AS average_order_value_brl,
    ROUND(o.otdr_pct, 2) AS on_time_delivery_rate_pct,
    f.total_leads,
    f.total_sellers_won,
    ROUND(f.conversion_rate_pct, 2) AS seller_lead_conversion_rate_pct
FROM OrderMetrics o
CROSS JOIN RevenueMetrics r
CROSS JOIN FunnelMetrics f;
"""

pd.read_sql_query(q41, conn)

,total_orders,total_revenue_brl,average_order_value_brl,on_time_delivery_rate_pct,total_leads,total_sellers_won,seller_lead_conversion_rate_pct
0,96478,13591643.7,120.65,91.89,8000,842,10.53
